## Step 1. 
Calculate the PMI for all successive pairs (w1, w2) of words in the Brown corpus. 

Words (not word pairs!) that occur in the corpus less than 10 times should be ignored. 

List the 20 word pairs with the highest PMI value and the 20 word pairs with the lowest PMI value. 

In [1]:
# get NGramModel from Part 3

from collections import Counter, defaultdict # You may import more from collections if needed
import regex as re
import numpy as np


class NGramModel:
    def __init__(self, text, n, alpha=0.0):
        """
        Initialize the NGramModel with text and the value of n.
        """
        self.text = text
        self.n = n
        self.alpha = alpha  # Alpha value for additive smoothing
        # Added self.list_ngrams variable to store the ngrams as a list
        self.list_ngrams = []
        self.ngrams = {}
        self.probabilities = {}
        self.vocab = set()
        self.words: list[str] = []

    def tokenize(self) -> list:
        """
        Tokenize the text into words. 
        Fill in the code to split the text into a list of words.
        """
        # Split by whitespace
        tokens = self.text.split()

        return tokens

    def generate_ngrams(self, tokens: list) -> list:
        """
        Generate n-grams from the list of tokens.
        Fill in the code to create n-grams.
        Make sure to treat each sentence independently, include the <s> and </s> tokens.
        """
        # Reset the ngrams list
        self.list_ngrams = []

        # Treat each sentence independently
        sentence = []

        for token in tokens:
            # Start sentence when start of sentence token is encountered
            if token == "<s>":
                sentence = ["<s>"]
                continue
            sentence.append(token)

            if token == "</s>":
                sentence = []
                continue

            if len(sentence) >= self.n:
                # Create the n grams as tuples and add the ngram to the list of ngrams
                self.list_ngrams.append(tuple(sentence[-self.n:]))

        # The template provided for this assignment indicated that this function is meant to return 
        # a list, while self.ngrams is defined as a dictionary. Thus we changed the type hint to dict.
        return self.list_ngrams

    def count_frequencies(self) -> None:
        """
        Count the frequencies of each n-gram.
        Fill in the code to count n-gram occurrences.
        """
        # Use the list of ngrams to populate the dictionary
        for i in self.list_ngrams:
            if i in self.ngrams:
                self.ngrams[i] += 1
            else:
                self.ngrams[i] = 1

    def calculate_probabilities(self) -> None:
        """
        Calculate probabilities of each n-gram based on its frequency. Add alpha smoothing separately.
        """
        # Total number of ngrams in the dataset
        total_ngrams = sum(self.ngrams.values())
        # For smoothing, unique number of n grams
        V = len(self.ngrams)
        for ngram, count in self.ngrams.items():
            self.probabilities[ngram] = (count + self.alpha) / (total_ngrams + (self.alpha * V))

    def most_frequent_ngrams(self, top_n: int = 10) -> list:
        """
        Return the most frequent n-grams and their probabilities.
        """
        # Sort the frequency first
        sorted_ngrams = sorted(self.ngrams.items(), key=lambda x: x[1], reverse=True)[:top_n]
        # Sort the probabilities based on the frequency
        sorted_grams = [(ngram, self.probabilities.get(ngram, 0)) for ngram,_ in sorted_ngrams]
    
        return sorted_grams
    
    def retrieve_words_and_vocab(self) -> None:
        '''Note that the words are lowercased and stripped of punctuation.'''

        words_POS = self.text.split()
        words = [word.rsplit('/', 1)[0] for word in words_POS] # max 2 items; take first item only

        mask = [bool(re.search(r'[a-zA-Z0-9]', word)) for word in words]
        words_array = np.array(words)
        filtered_words = words_array[mask]

        self.words = [re.sub(r'[^a-zA-Z0-9]', '', word).lower() for word in filtered_words]
        self.vocab = set(self.words)

    def return_valid_unigrams(self, n_valid: int = 10) -> set[str]:
        ''' return set of all words appearing at least n_valid times. Only call on unigram model. '''
        if self.n==1:
            valid_words = {ngram[0] for ngram, count in self.ngrams.items() if count >= n_valid}
            return valid_words
        else:
            raise ValueError("return_valid_unigrams can only be called on a unigram model (n=1)")
    
    def return_corpus_size(self) -> int:
        return len(self.words)



In [ ]:


import nltk
nltk.download('brown')
brown_corpus = nltk.corpus.brown
brown_text = brown_corpus.raw()

# clean raw text
words_POS = brown_text.split()
clean = [word.rsplit('/', 1)[0] for word in words_POS] # max 2 items; take first item only
removed_punct = [re.sub(r'[^a-zA-Z0-9]', '', word) for word in clean]
brown_text = ' '.join(removed_punct)


# unigram model
unigram_model = NGramModel(brown_text, 1)
tokens = unigram_model.tokenize()
unigram_model.generate_ngrams(tokens)
unigram_model.count_frequencies()
unigram_model.calculate_probabilities()
unigram_model.retrieve_words_and_vocab()

# bigram model for (w1,w2)
bigram_model = NGramModel(brown_text, 2)
bigram_model.generate_ngrams(tokens)  # reuse tokens
bigram_model.count_frequencies()
bigram_model.calculate_probabilities()

# total word count and retrieve valid words
N = unigram_model.return_corpus_size()
valid_words = unigram_model.return_valid_unigrams()

def freq(w) -> int:
    ''' returns absolute frequency of word '''
    return unigram_model.ngrams.get((w,), 0) # returns 0 if not found

def pmi(valid_words, n) -> list[tuple]:
    pmi_scores = []
    for w1 in valid_words:
        for w2 in valid_words:
            C12 = bigram_model.ngrams.get((w1, w2), 0)
            if C12 == 0:  # these would cause error with log(0)
                continue
            C1 = freq(w1)
            C2 = freq(w2)
            score = np.log((C12 * n) / (C1 * C2))
            pmi_scores.append(((w1, w2), score))
    return pmi_scores

pmi_scores = pmi(valid_words, N)
sorted_pmi = sorted(pmi_scores, key=lambda x: -x[1])
print("Top 20 highest PMI:", sorted_pmi[:20])
print("Top 20 lowest PMI:", sorted_pmi[-20:])



[nltk_data] Downloading package brown to /Users/Gileesa/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Top 20 highest PMI: [(('Hong', 'Kong'), np.float64(11.430846367079043)), (('Viet', 'Nam'), np.float64(11.056152917637634)), (('Pathet', 'Lao'), np.float64(10.995528295821199)), (('Simms', 'Purdew'), np.float64(10.995528295821199)), (('El', 'Paso'), np.float64(10.833009366323424)), (('7th', 'Cavalry'), np.float64(10.833009366323424)), (('Herald', 'Tribune'), np.float64(10.81180715867282)), (('Lo', 'Shu'), np.float64(10.784219202153992)), (('WTV', 'antigen'), np.float64(10.724795781683191)), (('Gray', 'Eyes'), np.float64(10.699477973698901)), (('Internal', 'Revenue'), np.float64(10.650687809529469)), (('Puerto', 'Rico'), np.float64(10.650687809529469)), (('decomposition', 'theorem'), np.float64(10.496537129702212)), (('Saxon', 'Shore'), np.float64(10.47883755260281)), (('anionic', 'binding'), np.float64(10.476334422384692)), (('ExportImport', 'Bank'), np.float64(10.461445809890941)), (('carbon', 'tetrachloride'), np.float64(10.442469908431935)), (('Common', 'Market'), np.float64(10.42754